<a href="https://colab.research.google.com/github/sttr47/Product-Review-Sentiment-Classifier/blob/main/Product%20Review%20Sentiment%20Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

ds = load_dataset("Yelp/yelp_review_full", split="train")

df = ds.shuffle(seed=42).select(range(12000)).to_pandas()

def to_sentiment(label):
    if label <= 1: return "negative"
    if label == 2: return "neutral"
    return "positive"

df["sentiment"] = df["label"].map(to_sentiment)

train, test = train_test_split(df, test_size=0.2, random_state=42,
                               stratify=df["sentiment"])
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

print(df["sentiment"].value_counts(normalize=True))

README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

sentiment
negative    0.403417
positive    0.401167
neutral     0.195417
Name: proportion, dtype: float64


In [2]:
import pandas as pd, numpy as np, time, re

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

labels   = ["negative", "neutral", "positive"]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

y_train = train["sentiment"].map(label2id).values
y_test  = test["sentiment"].map(label2id).values

def clean_tfidf(t):
    t = t.replace("\\n", " ").lower()
    t = re.sub(r"http\S+|<[^>]+>", " ", t) # remove links and HTML tags
    t = re.sub(r"\s+", " ", t).strip() # normalize whitespace
    return t

train["clean"] = train["text"].apply(clean_tfidf)
test["clean"]  = test["text"].apply(clean_tfidf)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
X_train = vec.fit_transform(train["clean"])

clf = LogisticRegression(max_iter=1000, C=2, class_weight="balanced")
clf.fit(X_train, y_train)

t0 = time.time()
X_test   = vec.transform(test["clean"])
proba_lr = clf.predict_proba(X_test)
lr_time  = time.time() - t0
pred_lr  = proba_lr.argmax(1)

print(f"Logistic Regression inference time: {lr_time:.3f}s")
print(classification_report(y_test, pred_lr, target_names=labels, digits=3))

Logistic Regression inference time: 0.983s
              precision    recall  f1-score   support

    negative      0.838     0.826     0.832       968
     neutral      0.516     0.539     0.528       469
    positive      0.826     0.819     0.823       963

    accuracy                          0.767      2400
   macro avg      0.727     0.728     0.727      2400
weighted avg      0.770     0.767     0.769      2400



In [4]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def make_ds(df, y):
    d = Dataset.from_dict({"text": df["text"].str.replace("\\n", " ", regex=False).tolist(), "labels": y.tolist()})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True, remove_columns=["text"])

train_ds = make_ds(train, y_train)
test_ds  = make_ds(test,  y_test)

cw = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        lab = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(out.logits.device))(out.logits, lab)
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="distilbert_yelp",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, data_collator=DataCollatorWithPadding(tok))
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.642195


TrainOutput(global_step=900, training_loss=0.5538900078667535, metrics={'train_runtime': 186.5969, 'train_samples_per_second': 154.343, 'train_steps_per_second': 4.823, 'total_flos': 1907440368344640.0, 'train_loss': 0.5538900078667535, 'epoch': 3.0})

In [6]:
from torch.nn.functional import softmax

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def predict(texts, bs=64):
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i+bs], truncation=True, max_length=256,
                      padding=True, return_tensors="pt").to(device)
            probs.append(softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(probs)

texts = test["text"].str.replace("\\n", " ", regex=False).tolist()

predict(texts[:64])
if device == "cuda": torch.cuda.synchronize()
t0 = time.time()
proba_bert = predict(texts)
if device == "cuda": torch.cuda.synchronize()
bert_time = time.time() - t0
pred_bert = proba_bert.argmax(1)

print(f"DistilBERT inference time: {bert_time:.2f}s")
print(classification_report(y_test, pred_bert, target_names=labels, digits=3))

DistilBERT inference time: 8.34s
              precision    recall  f1-score   support

    negative      0.888     0.826     0.856       968
     neutral      0.497     0.650     0.563       469
    positive      0.887     0.815     0.850       963

    accuracy                          0.787      2400
   macro avg      0.757     0.764     0.756      2400
weighted avg      0.811     0.787     0.796      2400



In [7]:
rows = []
for name, pred, t in [("TF-IDF + LR", pred_lr, lr_time), ("DistilBERT", pred_bert, bert_time)]:
    f1s = f1_score(y_test, pred, average=None)
    rows.append({"model": name,
                 "F1 negative": f1s[0],
                 "F1 neutral": f1s[1],
                 "F1 positive": f1s[2],
                 "macro-F1": f1s.mean(),
                 "accuracy": (pred == y_test).mean(),
                 "inference time": t})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# for name, pred in [("LR", pred_lr), ("BERT", pred_bert)]:
#     print(f"\n{name} confusion matrix (sətir=həqiqi, sütun=proqnoz):")
#     print(pd.DataFrame(confusion_matrix(y_test, pred), index=labels, columns=labels))

      model  F1 negative  F1 neutral  F1 positive  macro-F1  accuracy  inference time
TF-IDF + LR        0.832       0.528        0.823     0.727     0.768           0.983
 DistilBERT        0.856       0.563        0.850     0.756     0.788           8.341


In [10]:
res = test[["text", "sentiment"]].copy()
res["lr"]   = [id2label[i] for i in pred_lr]
res["bert"] = [id2label[i] for i in pred_bert]
res["bert_conf"] = proba_bert.max(1)
res.to_csv("predictions.csv", index=False)

lr_wrong   = res["lr"]   != res["sentiment"]
bert_wrong = res["bert"] != res["sentiment"]
both = res[lr_wrong & bert_wrong]

print("LR errors:      ", lr_wrong.sum())
print("BERT errors:    ", bert_wrong.sum())
print("Both wrong:     ", len(both))

picks = pd.concat([
    both[both.sentiment == "positive"].sample(1, random_state=1),
    both[both.sentiment == "negative"].sample(1, random_state=1),
    both[both.sentiment == "neutral"].sample(5, random_state=1),
])

for _, r in picks.iterrows():
    print(f"\nTrue={r.sentiment} | LR={r.lr} | BERT={r.bert} ({r.bert_conf:.2f})")
    print(r.text[:400].replace("\\n", " "))

LR errors:       558
BERT errors:     510
Both wrong:      311

True=positive | LR=neutral | BERT=neutral (0.73)
Been back a couple times since last review.  Margaritas were perfect both times.  Watermelon margaritas are unstoppable!  Chile relleno was a little bland and soupy, but the rest of the menu was terrific as usual.

True=negative | LR=neutral | BERT=neutral (0.42)
--LOOK AT MY PICTURE!!!   This is a picture of an **INSECT that I was about to eat!! I was really hungry and decide to stop by and eat.. Thank goodness this was my second plate and I wasn't too full because I gulped down the first plate.. This is your soup of toscana that I was served.   Management provided a friendly compensation, but I don't want anyone to get poisoned, sooo yeah, eat at you

True=neutral | LR=positive | BERT=positive (0.57)
We had a hasty brunch here right before we made a mad dash for the airport because 5Church happened to be open and convenient.  The standout for me was the bottomless sangria,

In [12]:
from sklearn.metrics import classification_report, f1_score

mask = y_test != 1
yb = (y_test[mask] == 2).astype(int)

def binary_eval(name, proba):
    p = proba[mask][:, [0, 2]]
    pred = p.argmax(1)
    print("Test without neutral")
    print(f"{name}")
    print(classification_report(yb, pred, target_names=["negative", "positive"], digits=3))
    return f1_score(yb, pred, average="macro"), (pred == yb).mean()

f1_lr_bin,   acc_lr_bin   = binary_eval("TF-IDF + Logistic Regression", proba_lr)
f1_bert_bin, acc_bert_bin = binary_eval("DistilBERT",  proba_bert)

print("\nComparison")
print(f"3 class : LR macro-F1={f1_score(y_test, pred_lr, average='macro'):.3f} | "
      f"DistilBERT F1={f1_score(y_test, pred_bert, average='macro'):.3f}")
print(f"2 class : Logistic Regression F1={f1_lr_bin:.3f} | "
      f"DistilBERT F1={f1_bert_bin:.3f}")

Test without neutral
TF-IDF + Logistic Regression
              precision    recall  f1-score   support

    negative      0.915     0.917     0.916       968
    positive      0.917     0.915     0.916       963

    accuracy                          0.916      1931
   macro avg      0.916     0.916     0.916      1931
weighted avg      0.916     0.916     0.916      1931

Test without neutral
DistilBERT
              precision    recall  f1-score   support

    negative      0.941     0.945     0.943       968
    positive      0.945     0.941     0.943       963

    accuracy                          0.943      1931
   macro avg      0.943     0.943     0.943      1931
weighted avg      0.943     0.943     0.943      1931


Comparison
3 class : LR macro-F1=0.727 | DistilBERT F1=0.756
2 class : Logistic Regression F1=0.916 | DistilBERT F1=0.943
